In [2]:
import os
import sys
import pandas as pd
from pathlib import Path
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

# Change root directory to the repo root (Jupyter: __file__ is not defined)
def find_repo_root(start=Path.cwd()):
	for p in [start] + list(start.parents):
		if (p / 'Code').exists() or (p / '.git').exists() or (p / 'local_repo').exists():
			return p
	return start

repo_root = find_repo_root()
data_root = repo_root.parent.parent / 'Data' / 'CBOS ready'
geo_root = repo_root.parent.parent / 'Data' / 'Geospatial'
gus_root = Path(os.getcwd()).parent.parent.parent.parent / "Data" / "GUS"

os.chdir(repo_root)
sys.path.append(str(repo_root / 'Code' / 'tools'))

# import local toolkit (try normal import first, fall back to loading from file)
try:
	import inequality_analyzers as inqA
	import local_utility_functions as luf
except Exception:
	import importlib.util
	toolkit_path = repo_root / 'Code' / 'tools' / 'inequality_analyzers.py'
	if toolkit_path.exists():
		spec = importlib.util.spec_from_file_location("inequality_analyzers", str(toolkit_path))
		stk = importlib.util.module_from_spec(spec)
		spec.loader.exec_module(stk)
	else:
		raise


In [3]:
# Load df and df_units from previously saved CSVs
df = pd.read_csv(gus_root / "metadata" / "bdl_variables_level6.csv", encoding="utf-8")
df_units = pd.read_csv(gus_root / "metadata" / "bdl_units_meta.csv", encoding="utf-8")

In [4]:
prg_05 = geo_root / "geometry" / "PRG_jednostki_administracyjne_2005"

# Read Obszary.shp file

gdf_2005 = gpd.read_file(prg_05 / "Obszary.shp", encoding="utf-8")
gdf_2005 = gdf_2005.to_crs(epsg=2180)  # Convert to EPSG:2180
#gdf_2005.plot()

In [5]:
prg_17 = geo_root / "geometry" / "PRG_jednostki_administracyjne_2017"

# Read gminy.shp file with polish characters

gdf_2017 = gpd.read_file(prg_17 / "gminy.shp")
gdf_2017 = gdf_2017.to_crs(epsg=2180)  # Convert to EPSG:2180
#gdf_2017.plot()

In [6]:
# Merging gdf_2017 and df_units on "jpt_kod_je" from gdf_2017 and "id" from df_units
# Creating columns "unitId" and "unitName" in gdf_2017 and "jpt_kod_je" in df_units for merging
gdf_2017['unitId'] = np.nan
gdf_2017['unitName'] = np.nan
df_units['jpt_kod_je'] = np.nan
for index, row in df_units.iterrows():
    gdf_code_from_units_id = luf.nuts_code_to_teryt(str(row['id']))
    where = np.where(gdf_2017['jpt_kod_je'].values == gdf_code_from_units_id)[0]
    if len(where) > 0:
        gdf_2017.at[where[0], 'unitId'] = row['id']
        gdf_2017.at[where[0], 'unitName'] = row['name']
        df_units.at[index, 'jpt_kod_je'] = gdf_code_from_units_id
    

/var/folders/y8/4_9g68pj7k136q2yypgp5ysc0000gn/T/ipykernel_62628/2389908868.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Bochnia' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  gdf_2017.at[where[0], 'unitName'] = row['name']
/var/folders/y8/4_9g68pj7k136q2yypgp5ysc0000gn/T/ipykernel_62628/2389908868.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1201011' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_units.at[index, 'jpt_kod_je'] = gdf_code_from_units_id


In [7]:
df_units

,id,name,level,hasDescription,parentId,kind,years_available,early_year,late_year,number_of_years,description,jpt_kod_je
0,0,POLSKA,0,False,NaN,NaN,[],NaN,NaN,0,NaN,NaN
1,10000000000,MAKROREGION POŁUDNIOWY,1,False,0.000000e+00,NaN,"[1995, 1996, 1997, 1998, 1999, 2000, 2001, 200...",1995.0,2026.0,32,NaN,NaN
2,11200000000,MAŁOPOLSKIE,2,True,1.000000e+10,NaN,"[1995, 1996, 1997, 1998, 1999, 2000, 2001, 200...",1995.0,2026.0,32,Zmiana granic województwa z dniem 01.01.2002 r...,NaN
3,11212001000,Powiat bocheński,5,False,1.121200e+10,1.0,"[1995, 1996, 1997, 1998, 1999, 2000, 2001, 200...",1995.0,2026.0,32,NaN,NaN
4,11212001011,Bochnia,6,False,1.121200e+10,1.0,"[1995, 1996, 1997, 1998, 1999, 2000, 2001, 200...",1995.0,2026.0,32,NaN,1201011
...,...,...,...,...,...,...,...,...,...,...,...,...
4599,71427338042,Radziejowice,6,False,7.142734e+10,2.0,"[1995, 1996, 1997, 1998, 1999, 2000, 2001, 200...",1995.0,2026.0,32,NaN,1438042
4600,71427338052,Wiskitki,6,True,7.142734e+10,2.0,"[1995, 1996, 1997, 1998, 1999, 2000, 2001, 200...",1995.0,2020.0,26,Zmiana rodzaju gminy z wiejskiego na miejsko-w...,1438052
4601,71427338053,Wiskitki,6,True,7.142734e+10,3.0,"[2021, 2022, 2023, 2024, 2025, 2026]",2021.0,2026.0,6,Zmiana rodzaju gminy z wiejskiego na miejsko-w...,NaN
4602,71427338054,Wiskitki - miasto,6,True,7.142734e+10,4.0,"[2021, 2022, 2023, 2024, 2025, 2026]",2021.0,2026.0,6,Zmiana rodzaju gminy z wiejskiego na miejsko-w...,NaN


In [8]:
def encode_level(row, col_names=['WOJ', 'POW', 'GMI']):
    c0, c1, c2 = col_names
    woj = row[c0]
    pow_ = row[c1]
    gmi = row[c2]
    if woj != '00' and pow_ == '00' and gmi == '00':
        return 2  # Voivodeship
    elif woj != '00' and pow_ != '00' and gmi == '00':
        return 5  # County
    elif woj != '00' and pow_ != '00' and gmi != '00':
        return 6  # Municipality
    else:
        return np.nan  # Undefined level

def encode_kind(row, col_name='RODZ'):
    val = str(row[col_name])
    if val == '0':
        return np.nan  # Not applicable
    mapping = {
        '1': 'urban',
        '2': 'rural',
        '3': 'urban-rural',
        '4': 'town',
        '5': 'village',
        '8': 'Warsaw district',
        '9': 'del. or district of a city'
    }
    return mapping.get(val, 'unknown')


In [9]:
# Oldest TERYT: TERC_Urzedowy_1999-01-01
# Newest TERYT: TERC_Urzedowy_2023-12-31

terc_1999 = pd.read_csv(geo_root / "TERC_Urzedowy_1999-01-01" / "TERC_Urzedowy_1999-01-01.csv", encoding="utf-8", sep=";")
terc_2024 = pd.read_csv(geo_root / "TERC_Urzedowy_2024-01-01" / "TERC_Urzedowy_2024-01-01.csv", encoding="utf-8", sep=";")

# File with all changes in TERYT codes in xml format
terc_changes = pd.read_xml(geo_root / "TERC_Urzedowy_zmiany_1999-01-01_2024-01-01.xml", encoding="utf-8")

In [10]:
# Change the columns 0, 1, 2 to string type with leading zeros of total length 2, 2, 2 respectively
# for 1999
terc_1999['WOJ'] = terc_1999['WOJ'].apply(lambda x: str(int(x)).zfill(2) if not pd.isna(x) else "00")
terc_1999['POW'] = terc_1999['POW'].apply(lambda x: str(int(x)).zfill(2) if not pd.isna(x) else "00")
terc_1999['GMI'] = terc_1999['GMI'].apply(lambda x: str(int(x)).zfill(2) if not pd.isna(x) else "00")
terc_1999['RODZ'] = terc_1999['RODZ'].apply(lambda x: str(int(x)).zfill(1) if not pd.isna(x) else "0")
terc_1999['level'] = terc_1999.apply(encode_level, axis=1)
terc_1999['kind'] = terc_1999.apply(encode_kind, axis=1)

# for 2023
terc_2024['WOJ'] = terc_2024['WOJ'].apply(lambda x: str(int(x)).zfill(2) if not pd.isna(x) else "00")
terc_2024['POW'] = terc_2024['POW'].apply(lambda x: str(int(x)).zfill(2) if not pd.isna(x) else "00")
terc_2024['GMI'] = terc_2024['GMI'].apply(lambda x: str(int(x)).zfill(2) if not pd.isna(x) else "00")
terc_2024['RODZ'] = terc_2024['RODZ'].apply(lambda x: str(int(x)).zfill(1) if not pd.isna(x) else "0")
terc_2024['level'] = terc_2024.apply(encode_level, axis=1)
terc_2024['kind'] = terc_2024.apply(encode_kind, axis=1)

In [11]:
# Create full TERYT code by concatenating the columns
terc_2024['id'] = terc_2024['WOJ'] + terc_2024['POW'] + terc_2024['GMI'] + terc_2024['RODZ']
terc_1999['id'] = terc_1999['WOJ'] + terc_1999['POW'] + terc_1999['GMI'] + terc_1999['RODZ']

In [12]:
terc_changes['WojPrzed'] = terc_changes['WojPrzed'].apply(lambda x: str(int(x)).zfill(2) if not pd.isna(x) else "00")
terc_changes['PowPrzed'] = terc_changes['PowPrzed'].apply(lambda x: str(int(x)).zfill(2) if not pd.isna(x) else "00")
terc_changes['GmiPrzed'] = terc_changes['GmiPrzed'].apply(lambda x: str(int(x)).zfill(2) if not pd.isna(x) else "00")
terc_changes['RodzPrzed'] = terc_changes['RodzPrzed'].apply(lambda x: str(int(x)).zfill(1) if not pd.isna(x) else "0")
terc_changes['WojPo'] = terc_changes['WojPo'].apply(lambda x: str(int(x)).zfill(2) if not pd.isna(x) else "00")
terc_changes['PowPo'] = terc_changes['PowPo'].apply(lambda x: str(int(x)).zfill(2) if not pd.isna(x) else "00")
terc_changes['GmiPo'] = terc_changes['GmiPo'].apply(lambda x: str(int(x)).zfill(2) if not pd.isna(x) else "00")
terc_changes['RodzPo'] = terc_changes['RodzPo'].apply(lambda x: str(int(x)).zfill(1) if not pd.isna(x) else "0")
terc_changes['id_before'] = terc_changes['WojPrzed'] + terc_changes['PowPrzed'] + terc_changes['GmiPrzed'] + terc_changes['RodzPrzed']
terc_changes['id_after'] = terc_changes['WojPo'] + terc_changes['PowPo'] + terc_changes['GmiPo'] + terc_changes['RodzPo']

In [13]:
terc_changes

,TypKorekty,WojPrzed,PowPrzed,GmiPrzed,RodzPrzed,NazwaPrzed,NazwaDodatkowaPrzed,StanPrzed,WojPo,PowPo,...,NazwaDodatkowaPo,WyodrebnionoZIdentyfikatora1,WyodrebnionoZIdentyfikatora2,WyodrebnionoZIdentyfikatora3,WlaczonoDoIdentyfikatora1,WlaczonoDoIdentyfikatora2,WlaczonoDoIdentyfikatora3,StanPo,id_before,id_after
0,M,02,20,02,2,Prusice,gmina wiejska,1999-01-01,02,20,...,gmina miejsko-wiejska,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-01,0220022,0220023
1,D,00,00,00,0,None,None,1999-01-01,02,20,...,miasto,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-01,0000000,0220024
2,D,00,00,00,0,None,None,1999-01-01,02,20,...,obszar wiejski,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-01,0000000,0220025
3,M,06,18,12,2,Tyszowce,gmina wiejska,1999-01-01,06,18,...,gmina miejsko-wiejska,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-01,0618122,0618123
4,D,00,00,00,0,None,None,1999-01-01,06,18,...,miasto,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-01,0000000,0618124
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
561,D,00,00,00,0,None,None,2023-01-01,30,08,...,miasto,NaN,NaN,NaN,NaN,NaN,NaN,2024-01-01,0000000,3008064
562,D,00,00,00,0,None,None,2023-01-01,30,08,...,obszar wiejski,NaN,NaN,NaN,NaN,NaN,NaN,2024-01-01,0000000,3008065
563,M,30,28,04,2,Mieścisko,gmina wiejska,2023-01-01,30,28,...,gmina miejsko-wiejska,NaN,NaN,NaN,NaN,NaN,NaN,2024-01-01,3028042,3028043
564,D,00,00,00,0,None,None,2023-01-01,30,28,...,miasto,NaN,NaN,NaN,NaN,NaN,NaN,2024-01-01,0000000,3028044


In [14]:
# Let's explore the terc_changes columns and TypKorekty values in detail
print("Columns:", terc_changes.columns.tolist())
print("\nTypKorekty value counts:")
print(terc_changes['TypKorekty'].value_counts())
print("\nFirst few rows with all columns:")
terc_changes.head(10).T

Columns: ['TypKorekty', 'WojPrzed', 'PowPrzed', 'GmiPrzed', 'RodzPrzed', 'NazwaPrzed', 'NazwaDodatkowaPrzed', 'StanPrzed', 'WojPo', 'PowPo', 'GmiPo', 'RodzPo', 'NazwaPo', 'NazwaDodatkowaPo', 'WyodrebnionoZIdentyfikatora1', 'WyodrebnionoZIdentyfikatora2', 'WyodrebnionoZIdentyfikatora3', 'WlaczonoDoIdentyfikatora1', 'WlaczonoDoIdentyfikatora2', 'WlaczonoDoIdentyfikatora3', 'StanPo', 'id_before', 'id_after']

TypKorekty value counts:
TypKorekty
D    301
M    258
U      7
Name: count, dtype: int64

First few rows with all columns:


,0,1,2,3,4,5,6,7,8,9
TypKorekty,M,D,D,M,D,D,M,D,D,M
WojPrzed,02,00,00,06,00,00,12,00,00,12
PowPrzed,20,00,00,18,00,00,02,00,00,10
GmiPrzed,02,00,00,12,00,00,03,00,00,13
RodzPrzed,2,0,0,2,0,0,2,0,0,4
NazwaPrzed,Prusice,None,None,Tyszowce,None,None,Czchów,None,None,Piwniczna
NazwaDodatkowaPrzed,gmina wiejska,None,None,gmina wiejska,None,None,gmina wiejska,None,None,miasto
StanPrzed,1999-01-01,1999-01-01,1999-01-01,1999-01-01,1999-01-01,1999-01-01,1999-01-01,1999-01-01,1999-01-01,1999-01-01
WojPo,02,02,02,06,06,06,12,12,12,00
PowPo,20,20,20,18,18,18,02,02,02,00


In [15]:
# Explore change types in more detail
# D = Dodano (Added), M = Modyfikacja (Modification), U = Usunięto (Removed)

print("=== Type D (Added) examples ===")
display(terc_changes[terc_changes['TypKorekty'] == 'D'].head(5))

print("\n=== Type M (Modified) examples ===")
display(terc_changes[terc_changes['TypKorekty'] == 'M'].head(5))

print("\n=== Type U (Removed) examples ===")
display(terc_changes[terc_changes['TypKorekty'] == 'U'].head(5))

# Check the date columns
print("\n=== Unique years in StanPrzed ===")
terc_changes['year_before'] = pd.to_datetime(terc_changes['StanPrzed']).dt.year
terc_changes['year_after'] = pd.to_datetime(terc_changes['StanPo']).dt.year
print(terc_changes['year_after'].value_counts().sort_index())

=== Type D (Added) examples ===


,TypKorekty,WojPrzed,PowPrzed,GmiPrzed,RodzPrzed,NazwaPrzed,NazwaDodatkowaPrzed,StanPrzed,WojPo,PowPo,...,NazwaDodatkowaPo,WyodrebnionoZIdentyfikatora1,WyodrebnionoZIdentyfikatora2,WyodrebnionoZIdentyfikatora3,WlaczonoDoIdentyfikatora1,WlaczonoDoIdentyfikatora2,WlaczonoDoIdentyfikatora3,StanPo,id_before,id_after
1,D,00,00,00,0,None,None,1999-01-01,02,20,...,miasto,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-01,0000000,0220024
2,D,00,00,00,0,None,None,1999-01-01,02,20,...,obszar wiejski,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-01,0000000,0220025
4,D,00,00,00,0,None,None,1999-01-01,06,18,...,miasto,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-01,0000000,0618124
5,D,00,00,00,0,None,None,1999-01-01,06,18,...,obszar wiejski,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-01,0000000,0618125
7,D,00,00,00,0,None,None,1999-01-01,12,02,...,miasto,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-01,0000000,1202034



=== Type M (Modified) examples ===


,TypKorekty,WojPrzed,PowPrzed,GmiPrzed,RodzPrzed,NazwaPrzed,NazwaDodatkowaPrzed,StanPrzed,WojPo,PowPo,...,NazwaDodatkowaPo,WyodrebnionoZIdentyfikatora1,WyodrebnionoZIdentyfikatora2,WyodrebnionoZIdentyfikatora3,WlaczonoDoIdentyfikatora1,WlaczonoDoIdentyfikatora2,WlaczonoDoIdentyfikatora3,StanPo,id_before,id_after
0,M,02,20,02,2,Prusice,gmina wiejska,1999-01-01,02,20,...,gmina miejsko-wiejska,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-01,0220022,0220023
3,M,06,18,12,2,Tyszowce,gmina wiejska,1999-01-01,06,18,...,gmina miejsko-wiejska,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-01,0618122,0618123
6,M,12,02,03,2,Czchów,gmina wiejska,1999-01-01,12,02,...,gmina miejsko-wiejska,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-01,1202032,1202033
9,M,12,10,13,4,Piwniczna,miasto,1999-01-01,00,00,...,None,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-01,1210134,0000000
10,M,12,11,12,4,Rabka,miasto,1999-01-01,00,00,...,None,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-01,1211124,0000000



=== Type U (Removed) examples ===


,TypKorekty,WojPrzed,PowPrzed,GmiPrzed,RodzPrzed,NazwaPrzed,NazwaDodatkowaPrzed,StanPrzed,WojPo,PowPo,...,NazwaDodatkowaPo,WyodrebnionoZIdentyfikatora1,WyodrebnionoZIdentyfikatora2,WyodrebnionoZIdentyfikatora3,WlaczonoDoIdentyfikatora1,WlaczonoDoIdentyfikatora2,WlaczonoDoIdentyfikatora3,StanPo,id_before,id_after
128,U,14,31,00,0,warszawski,powiat,2002-01-01,00,00,...,None,NaN,NaN,NaN,NaN,NaN,NaN,2002-10-27,1431000,0000000
129,U,14,31,04,1,Warszawa-Centrum,gmina miejska,2002-01-01,00,00,...,None,NaN,NaN,NaN,NaN,NaN,NaN,2002-10-27,1431041,0000000
135,U,02,63,00,0,Wałbrzych,miasto na prawach powiatu,2002-10-27,00,00,...,None,NaN,NaN,NaN,NaN,NaN,NaN,2003-01-01,0263000,0000000
251,U,08,09,10,2,Zielona Góra,gmina wiejska,2014-01-01,00,00,...,None,NaN,NaN,NaN,NaN,NaN,NaN,2015-01-01,0809102,0000000
311,U,12,10,02,4,Chełmiec,miasto,2018-01-01,00,00,...,None,NaN,NaN,NaN,NaN,NaN,NaN,2018-01-01,1210024,0000000



=== Unique years in StanPrzed ===
year_after
2000     17
2001     12
2002    101
2003      6
2004     10
2005      4
2006      7
2007      6
2008      6
2009     15
2010     21
2011     16
2012      1
2013      2
2014     18
2015     10
2016     17
2017     16
2018     28
2019     31
2020     12
2021     32
2022     30
2023     45
2024    103
Name: count, dtype: int64


In [16]:
# Let's inspect the columns WyodrebnionoZIdentyfikatora and WlaczonoDoIdentyfikatora
# These indicate which units were separated from or merged into
print("=== Columns with extraction/merge info ===")
print("\nWyodrebnionoZIdentyfikatora1 (Separated from):")
print(terc_changes['WyodrebnionoZIdentyfikatora1'].dropna().head(10))
print("\nWlaczonoDoIdentyfikatora1 (Merged into):")
print(terc_changes['WlaczonoDoIdentyfikatora1'].dropna().head(10))

# Check the structure of terc_1999 and terc_2024
print("\n=== terc_1999 columns ===")
print(terc_1999.columns.tolist())
print(f"\nRows in terc_1999: {len(terc_1999)}")
print(f"Rows in terc_2024: {len(terc_2024)}")

# Sample of the terc_1999 data
print("\n=== terc_1999 sample (level 6 = gmina) ===")
display(terc_1999[terc_1999['level'] == 6].head(10))

=== Columns with extraction/merge info ===

WyodrebnionoZIdentyfikatora1 (Separated from):
Series([], Name: WyodrebnionoZIdentyfikatora1, dtype: float64)

WlaczonoDoIdentyfikatora1 (Merged into):
Series([], Name: WlaczonoDoIdentyfikatora1, dtype: float64)

=== terc_1999 columns ===
['WOJ', 'POW', 'GMI', 'RODZ', 'NAZWA', 'NAZWA_DOD', 'STAN_NA', 'level', 'kind', 'id']

Rows in terc_1999: 4038
Rows in terc_2024: 4332

=== terc_1999 sample (level 6 = gmina) ===


,WOJ,POW,GMI,RODZ,NAZWA,NAZWA_DOD,STAN_NA,level,kind,id
2,02,01,01,1,Bolesławiec,gmina miejska,1999-01-01,6,urban,0201011
3,02,01,02,2,Bolesławiec,gmina wiejska,1999-01-01,6,rural,0201022
4,02,01,03,2,Gromadka,gmina wiejska,1999-01-01,6,rural,0201032
5,02,01,04,3,Nowogrodziec,gmina miejsko-wiejska,1999-01-01,6,urban-rural,0201043
6,02,01,04,4,Nowogrodziec,miasto,1999-01-01,6,town,0201044
7,02,01,04,5,Nowogrodziec,obszar wiejski,1999-01-01,6,village,0201045
8,02,01,05,2,Osiecznica,gmina wiejska,1999-01-01,6,rural,0201052
9,02,01,06,2,Warta Bolesławiecka,gmina wiejska,1999-01-01,6,rural,0201062
11,02,02,01,1,Bielawa,gmina miejska,1999-01-01,6,urban,0202011
12,02,02,02,1,Dzierżoniów,gmina miejska,1999-01-01,6,urban,0202021


In [17]:
# Reload the local_utility_functions module to get the new functions
import importlib
importlib.reload(luf)

# Verify the new functions are available
print("Available functions in luf:")
print([f for f in dir(luf) if not f.startswith('_')])

Available functions in luf:
['Path', 'adapt_txt', 'apply_changes_to_teryt', 'classify_change', 'encode_kind', 'encode_level', 'get_changes_summary', 'get_level_name', 'get_unit_history', 'harmonize_teryt', 'np', 'nuts_code_to_teryt', 'pd', 'prepare_changes_dataframe', 'prepare_teryt_dataframe', 'remove_polish_characters']


In [18]:
# Test the harmonize_teryt function
# Note: terc_1999 and terc_2024 should already be loaded and preprocessed

# Run the harmonization
mega_df = luf.harmonize_teryt(terc_1999, terc_2024, terc_changes)

Harmonizing TERYT codes from 1999 to 2024...
  Year 2000: applying 17 changes...
  Year 2001: applying 12 changes...
  Year 2002: applying 101 changes...
  Year 2003: applying 6 changes...
  Year 2004: applying 10 changes...
  Year 2005: applying 4 changes...
  Year 2006: applying 7 changes...
  Year 2007: applying 6 changes...
  Year 2008: applying 6 changes...
  Year 2009: applying 15 changes...
  Year 2010: applying 21 changes...
  Year 2011: applying 16 changes...
  Year 2012: applying 1 changes...
  Year 2013: applying 2 changes...
  Year 2014: applying 18 changes...
  Year 2015: applying 10 changes...
  Year 2016: applying 17 changes...
  Year 2017: applying 16 changes...
  Year 2018: applying 28 changes...
  Year 2019: applying 31 changes...
  Year 2020: applying 12 changes...
  Year 2021: applying 32 changes...
  Year 2022: applying 30 changes...
  Year 2023: applying 45 changes...
  Year 2024: applying 103 changes...

Harmonization complete!
Total rows in mega DataFrame: 10734

In [19]:
# Test 1: Get division for a specific year (as specified in requirements)
division_2010 = mega_df[mega_df['year'] == 2010]
print(f"Administrative division in 2010:")
print(f"  Total units: {len(division_2010)}")
print(f"  Voivodeships: {len(division_2010[division_2010['level'] == 2])}")
print(f"  Powiats: {len(division_2010[division_2010['level'] == 5])}")
print(f"  Gminas: {len(division_2010[division_2010['level'] == 6])}")

# Test 2: Show sample of mega_df structure
print("\n=== Sample of mega_df ===")
display(mega_df.head(10))

Administrative division in 2010:
  Total units: 4105
  Voivodeships: 16
  Powiats: 379
  Gminas: 3710

=== Sample of mega_df ===


,year,WOJ,POW,GMI,RODZ,id,NAZWA,NAZWA_DOD,level,kind,STAN_NA,if_changed,when_changed,notes
0,1999,02,00,00,0,0200000,DOLNOŚLĄSKIE,województwo,2,NaN,1999-01-01,False,NaN,"{'number_of_changes': 0, 'changes': []}"
1,1999,02,01,00,0,0201000,bolesławiecki,powiat,5,NaN,1999-01-01,False,NaN,"{'number_of_changes': 0, 'changes': []}"
2,1999,02,01,01,1,0201011,Bolesławiec,gmina miejska,6,urban,1999-01-01,False,NaN,"{'number_of_changes': 0, 'changes': []}"
3,1999,02,01,02,2,0201022,Bolesławiec,gmina wiejska,6,rural,1999-01-01,False,NaN,"{'number_of_changes': 0, 'changes': []}"
4,1999,02,01,03,2,0201032,Gromadka,gmina wiejska,6,rural,1999-01-01,False,NaN,"{'number_of_changes': 0, 'changes': []}"
5,1999,02,01,04,3,0201043,Nowogrodziec,gmina miejsko-wiejska,6,urban-rural,1999-01-01,False,NaN,"{'number_of_changes': 0, 'changes': []}"
6,1999,02,01,04,4,0201044,Nowogrodziec,miasto,6,town,1999-01-01,False,NaN,"{'number_of_changes': 0, 'changes': []}"
7,1999,02,01,04,5,0201045,Nowogrodziec,obszar wiejski,6,village,1999-01-01,False,NaN,"{'number_of_changes': 0, 'changes': []}"
8,1999,02,01,05,2,0201052,Osiecznica,gmina wiejska,6,rural,1999-01-01,False,NaN,"{'number_of_changes': 0, 'changes': []}"
9,1999,02,01,06,2,0201062,Warta Bolesławiecka,gmina wiejska,6,rural,1999-01-01,False,NaN,"{'number_of_changes': 0, 'changes': []}"


In [20]:
# Test 3: Show units that were changed
print("=== Sample of changed units ===")
changed_units = mega_df[mega_df['if_changed'] == True]
print(f"Total entries with changes: {len(changed_units)}")
print(f"Unique changed unit IDs: {changed_units['id'].nunique()}")

# Show examples of different change types
print("\n=== Sample entries with 'notes' containing changes ===")
sample_changed = changed_units[changed_units['notes'].apply(
    lambda x: isinstance(x, dict) and len(x.get('changes', [])) > 0
)].head(10)
display(sample_changed[['year', 'id', 'NAZWA', 'when_changed', 'notes']])

=== Sample of changed units ===
Total entries with changes: 5869
Unique changed unit IDs: 556

=== Sample entries with 'notes' containing changes ===


,year,id,NAZWA,when_changed,notes
4251,2000,0220023,Prusice,2000.0,"{'number_of_changes': 1, 'changes': ['TYPE_CHA..."
4824,2000,0618123,Tyszowce,2000.0,"{'number_of_changes': 1, 'changes': ['TYPE_CHA..."
5300,2000,1202033,Czchów,2000.0,"{'number_of_changes': 1, 'changes': ['TYPE_CHA..."
5427,2000,1210134,Piwniczna-Zdrój,2000.0,"{'number_of_changes': 1, 'changes': ['TYPE_CHA..."
5447,2000,1211124,Rabka-Zdrój,2000.0,"{'number_of_changes': 1, 'changes': ['TYPE_CHA..."
5915,2000,1429053,Kosów Lacki,2000.0,"{'number_of_changes': 1, 'changes': ['TYPE_CHA..."
7806,2000,3030033,Nekla,2000.0,"{'number_of_changes': 1, 'changes': ['TYPE_CHA..."
8076,2000,0220024,Prusice,2000.0,"{'number_of_changes': 1, 'changes': ['NEW: gmi..."
8077,2000,0220025,Prusice,2000.0,"{'number_of_changes': 1, 'changes': ['NEW: gmi..."
8078,2000,0618124,Tyszowce,2000.0,"{'number_of_changes': 1, 'changes': ['NEW: gmi..."


In [21]:
# Test 4: Get changes summary
changes_summary = luf.get_changes_summary(mega_df)
print("=== Changes Summary by Year ===")
display(changes_summary)

# Test 5: Look at a specific unit's history
print("\n=== History of 'Prusice' gmina ===")
prusice_history = luf.get_unit_history(mega_df, unit_name='Prusice')
# Select only gmina level and exclude 'notes' for display
prusice_gmina = prusice_history[prusice_history['level'] == 6][['year', 'id', 'NAZWA', 'NAZWA_DOD', 'kind', 'if_changed', 'when_changed']]
display(prusice_gmina.head(30))

=== Changes Summary by Year ===


,year,change_count,change_types
0,2000,425,"[NEW: gmina created (id: 1202034), NEW: gmina ..."
1,2001,288,"[NEW: gmina created (id: 1412075), NEW: gmina ..."
2,2002,2229,"[GMINA_CODE_CHANGE: 07 -> 02, GMINA_CODE_CHANG..."
3,2003,98,"[POWIAT_CHANGE: 63 -> 21, GMINA_CODE_CHANGE: 1..."
4,2004,203,"[NEW: gmina created (id: 3207014), NEW: gmina ..."
5,2005,80,"[POWIAT_CHANGE: 02 -> 00, VOIVODESHIP_CHANGE: ..."
6,2006,125,"[NEW: gmina created (id: 1006105), GMINA_CODE_..."
7,2007,108,[DESIGNATION_CHANGE: gmina wiejska -> gmina mi...
8,2008,102,"[TYPE_CHANGE: urban -> urban-rural, DESIGNATIO..."
9,2009,240,"[NEW: gmina created (id: 1803025), NEW: gmina ..."



=== History of 'Prusice' gmina ===


,year,id,NAZWA,NAZWA_DOD,kind,if_changed,when_changed
213,1999,0220022,Prusice,gmina wiejska,rural,False,NaN
4251,2000,0220023,Prusice,gmina miejsko-wiejska,urban-rural,True,2000.0
8076,2000,0220024,Prusice,miasto,town,True,2000.0
8077,2000,0220025,Prusice,obszar wiejski,village,True,2000.0
8299,2001,0220023,Prusice,gmina miejsko-wiejska,urban-rural,True,2000.0
12124,2001,0220024,Prusice,miasto,town,True,2000.0
12125,2001,0220025,Prusice,obszar wiejski,village,True,2000.0
12355,2002,0220023,Prusice,gmina miejsko-wiejska,urban-rural,True,2000.0
16178,2002,0220024,Prusice,miasto,town,True,2000.0
16179,2002,0220025,Prusice,obszar wiejski,village,True,2000.0


In [22]:
# Test 6: Validate that the final year matches terc_2024
division_2024 = mega_df[mega_df['year'] == 2024]
print("=== Validation: 2024 Division ===")
print(f"Units in mega_df for 2024: {len(division_2024)}")
print(f"Units in terc_2024: {len(terc_2024)}")
print(f"\nBreakdown by level in mega_df 2024:")
print(f"  Voivodeships: {len(division_2024[division_2024['level'] == 2])}")
print(f"  Powiats: {len(division_2024[division_2024['level'] == 5])}")
print(f"  Gminas: {len(division_2024[division_2024['level'] == 6])}")

print(f"\nBreakdown by level in terc_2024:")
print(f"  Voivodeships: {len(terc_2024[terc_2024['level'] == 2])}")
print(f"  Powiats: {len(terc_2024[terc_2024['level'] == 5])}")
print(f"  Gminas: {len(terc_2024[terc_2024['level'] == 6])}")

# Check if final counts match
if len(division_2024) == len(terc_2024):
    print("\n✓ The 2024 division in mega_df matches terc_2024!")
else:
    print(f"\n⚠ Mismatch: mega_df has {len(division_2024)} units vs terc_2024 with {len(terc_2024)} units")
    print("This might be due to units being added/removed in the processing. Checking...")
    
    # Check ID differences
    mega_ids = set(division_2024['id'].unique())
    terc_ids = set(terc_2024['id'].unique())
    
    only_in_mega = mega_ids - terc_ids
    only_in_terc = terc_ids - mega_ids
    
    if only_in_mega:
        print(f"\nIDs only in mega_df (count: {len(only_in_mega)}):")
        print(list(only_in_mega)[:10])
    if only_in_terc:
        print(f"\nIDs only in terc_2024 (count: {len(only_in_terc)}):")
        print(list(only_in_terc)[:10])

=== Validation: 2024 Division ===
Units in mega_df for 2024: 4334
Units in terc_2024: 4332

Breakdown by level in mega_df 2024:
  Voivodeships: 16
  Powiats: 380
  Gminas: 3938

Breakdown by level in terc_2024:
  Voivodeships: 16
  Powiats: 380
  Gminas: 3936

⚠ Mismatch: mega_df has 4334 units vs terc_2024 with 4332 units
This might be due to units being added/removed in the processing. Checking...

IDs only in mega_df (count: 2):
['1210025', '1210024']


In [23]:
# Debug: Check what units have ID 0000000
debug_zero_ids = mega_df[mega_df['id'] == '0000000']
print(f"Units with id='0000000': {len(debug_zero_ids)}")
print("\nSample of these units:")
display(debug_zero_ids[['year', 'id', 'NAZWA', 'NAZWA_DOD', 'WOJ', 'POW', 'GMI', 'RODZ', 'notes']].head(10))

# Check the changes where id_before is 0000000 (i.e. new units being created)
debug_new_changes = terc_changes[terc_changes['id_before'] == '0000000']
print(f"\nChanges where id_before='0000000' (new units): {len(debug_new_changes)}")
print("Sample:")
display(debug_new_changes[['TypKorekty', 'id_before', 'id_after', 'NazwaPo', 'NazwaDodatkowaPo', 'StanPo']].head(10))

Units with id='0000000': 0

Sample of these units:


,year,id,NAZWA,NAZWA_DOD,WOJ,POW,GMI,RODZ,notes



Changes where id_before='0000000' (new units): 301
Sample:


,TypKorekty,id_before,id_after,NazwaPo,NazwaDodatkowaPo,StanPo
1,D,0000000,0220024,Prusice,miasto,2000-01-01
2,D,0000000,0220025,Prusice,obszar wiejski,2000-01-01
4,D,0000000,0618124,Tyszowce,miasto,2000-01-01
5,D,0000000,0618125,Tyszowce,obszar wiejski,2000-01-01
7,D,0000000,1202034,Czchów,miasto,2000-01-01
8,D,0000000,1202035,Czchów,obszar wiejski,2000-01-01
12,D,0000000,1429054,Kosów Lacki,miasto,2000-01-01
13,D,0000000,1429055,Kosów Lacki,obszar wiejski,2000-01-01
15,D,0000000,3030034,Nekla,miasto,2000-01-01
16,D,0000000,3030035,Nekla,obszar wiejski,2000-01-01


In [24]:
# Re-run with the fixed function
importlib.reload(luf)
mega_df_v2 = luf.harmonize_teryt(terc_1999, terc_2024, terc_changes)

Harmonizing TERYT codes from 1999 to 2024...
  Year 2000: applying 17 changes...
  Year 2001: applying 12 changes...
  Year 2002: applying 101 changes...
  Year 2003: applying 6 changes...
  Year 2004: applying 10 changes...
  Year 2005: applying 4 changes...
  Year 2006: applying 7 changes...
  Year 2007: applying 6 changes...
  Year 2008: applying 6 changes...
  Year 2009: applying 15 changes...
  Year 2010: applying 21 changes...
  Year 2011: applying 16 changes...
  Year 2012: applying 1 changes...
  Year 2013: applying 2 changes...
  Year 2014: applying 18 changes...
  Year 2015: applying 10 changes...
  Year 2016: applying 17 changes...
  Year 2017: applying 16 changes...
  Year 2018: applying 28 changes...
  Year 2019: applying 31 changes...
  Year 2020: applying 12 changes...
  Year 2021: applying 32 changes...
  Year 2022: applying 30 changes...
  Year 2023: applying 45 changes...
  Year 2024: applying 103 changes...

Harmonization complete!
Total rows in mega DataFrame: 10734

In [25]:
# Check if zero IDs are fixed
debug_zero_ids_v2 = mega_df_v2[mega_df_v2['id'] == '0000000']
print(f"Units with id='0000000' after fix: {len(debug_zero_ids_v2)}")

# Validate 2024 again
division_2024_v2 = mega_df_v2[mega_df_v2['year'] == 2024]
print(f"\n=== Validation: 2024 Division (v2) ===")
print(f"Units in mega_df for 2024: {len(division_2024_v2)}")
print(f"Units in terc_2024: {len(terc_2024)}")

# Check ID differences
mega_ids_v2 = set(division_2024_v2['id'].unique())
terc_ids = set(terc_2024['id'].unique())

only_in_mega_v2 = mega_ids_v2 - terc_ids
only_in_terc = terc_ids - mega_ids_v2

print(f"\nIDs only in mega_df: {len(only_in_mega_v2)}")
print(f"IDs only in terc_2024: {len(only_in_terc)}")

Units with id='0000000' after fix: 0

=== Validation: 2024 Division (v2) ===
Units in mega_df for 2024: 4334
Units in terc_2024: 4332

IDs only in mega_df: 2
IDs only in terc_2024: 0


In [26]:
# Check what type of changes have id_before='0000000'
changes_with_zero = terc_changes[terc_changes['id_before'] == '0000000']
print(f"Changes with id_before='0000000' by type:")
print(changes_with_zero['TypKorekty'].value_counts())

# Check M type changes with zero before
m_changes_zero = changes_with_zero[changes_with_zero['TypKorekty'] == 'M']
print(f"\nM-type changes with id_before='0000000': {len(m_changes_zero)}")
if len(m_changes_zero) > 0:
    print("\nSample:")
    display(m_changes_zero[['TypKorekty', 'id_before', 'id_after', 'NazwaPrzed', 'NazwaPo', 'NazwaDodatkowaPrzed', 'NazwaDodatkowaPo', 'StanPo']].head(10))

Changes with id_before='0000000' by type:
TypKorekty
D    301
Name: count, dtype: int64

M-type changes with id_before='0000000': 0


In [27]:
# Look at the timeline of a specific unit with ID=0000000
unit_zero_sample = mega_df_v2[mega_df_v2['id'] == '0000000']
print("Sample of units with id='0000000':")
sample_names = unit_zero_sample['NAZWA'].unique()[:5]
for name in sample_names:
    print(f"\n--- History of '{name}' ---")
    unit_hist = mega_df_v2[mega_df_v2['NAZWA'].str.contains(name, na=False)]
    # Show only the first occurrence per year
    display(unit_hist[['year', 'id', 'NAZWA', 'NAZWA_DOD', 'WOJ', 'POW', 'GMI', 'RODZ', 'if_changed']].drop_duplicates(['year', 'NAZWA_DOD']).head(10))

Sample of units with id='0000000':


In [28]:
# Check the original changes for Piwniczna-Zdrój 
piwniczna_changes = terc_changes[terc_changes['NazwaPo'].str.contains('Piwniczna', na=False) | 
                                  terc_changes['NazwaPrzed'].str.contains('Piwniczna', na=False)]
print("Changes involving 'Piwniczna':")
display(piwniczna_changes[['TypKorekty', 'NazwaPrzed', 'NazwaDodatkowaPrzed', 'id_before', 
                            'NazwaPo', 'NazwaDodatkowaPo', 'id_after', 'StanPo']])

Changes involving 'Piwniczna':


,TypKorekty,NazwaPrzed,NazwaDodatkowaPrzed,id_before,NazwaPo,NazwaDodatkowaPo,id_after,StanPo
9,M,Piwniczna,miasto,1210134,Piwniczna-Zdrój,None,0000000,2000-01-01
48,M,Piwniczna,gmina miejsko-wiejska,1210133,Piwniczna-Zdrój,None,0000000,2002-01-01
49,M,Piwniczna,obszar wiejski,1210135,Piwniczna-Zdrój,None,0000000,2002-01-01


In [29]:
# Re-run with the updated fix
importlib.reload(luf)
mega_df_v3 = luf.harmonize_teryt(terc_1999, terc_2024, terc_changes)

# Validate
debug_zero_ids_v3 = mega_df_v3[mega_df_v3['id'] == '0000000']
print(f"\nUnits with id='0000000' after second fix: {len(debug_zero_ids_v3)}")

# Validate 2024
division_2024_v3 = mega_df_v3[mega_df_v3['year'] == 2024]
print(f"\n=== Validation: 2024 Division (v3) ===")
print(f"Units in mega_df for 2024: {len(division_2024_v3)}")
print(f"Units in terc_2024: {len(terc_2024)}")

Harmonizing TERYT codes from 1999 to 2024...
  Year 2000: applying 17 changes...
  Year 2001: applying 12 changes...
  Year 2002: applying 101 changes...
  Year 2003: applying 6 changes...
  Year 2004: applying 10 changes...
  Year 2005: applying 4 changes...
  Year 2006: applying 7 changes...
  Year 2007: applying 6 changes...
  Year 2008: applying 6 changes...
  Year 2009: applying 15 changes...
  Year 2010: applying 21 changes...
  Year 2011: applying 16 changes...
  Year 2012: applying 1 changes...
  Year 2013: applying 2 changes...
  Year 2014: applying 18 changes...
  Year 2015: applying 10 changes...
  Year 2016: applying 17 changes...
  Year 2017: applying 16 changes...
  Year 2018: applying 28 changes...
  Year 2019: applying 31 changes...
  Year 2020: applying 12 changes...
  Year 2021: applying 32 changes...
  Year 2022: applying 30 changes...
  Year 2023: applying 45 changes...
  Year 2024: applying 103 changes...

Harmonization complete!
Total rows in mega DataFrame: 10734

In [30]:
# Final validation - check the differences
mega_ids_v3 = set(division_2024_v3['id'].unique())
terc_ids = set(terc_2024['id'].unique())

only_in_mega_v3 = mega_ids_v3 - terc_ids
only_in_terc = terc_ids - mega_ids_v3

print(f"IDs only in mega_df (count: {len(only_in_mega_v3)}):")
if only_in_mega_v3:
    extra_units = division_2024_v3[division_2024_v3['id'].isin(only_in_mega_v3)]
    display(extra_units[['id', 'NAZWA', 'NAZWA_DOD', 'WOJ', 'POW', 'GMI', 'RODZ']])

print(f"\nIDs only in terc_2024 (count: {len(only_in_terc)}):")
if only_in_terc:
    missing_units = terc_2024[terc_2024['id'].isin(list(only_in_terc)[:10])]
    display(missing_units[['id', 'NAZWA', 'NAZWA_DOD', 'WOJ', 'POW', 'GMI', 'RODZ']].head(10))

IDs only in mega_df (count: 2):


,id,NAZWA,NAZWA_DOD,WOJ,POW,GMI,RODZ
107168,1210024,Chełmiec,miasto,12,10,02,4
107169,1210025,Chełmiec,obszar wiejski,12,10,02,5



IDs only in terc_2024 (count: 0):


## Final Results: TERYT Harmonization Summary

The harmonization is complete! Here's how to use the mega DataFrame:

In [31]:
# Usage examples for the harmonized TERYT mega DataFrame

print("=" * 60)
print("TERYT HARMONIZATION - USAGE EXAMPLES")
print("=" * 60)

# 1. Get administrative division for a specific year
print("\n1. Get administrative division for a specific year:")
print("   division_2010 = mega_df_v3[mega_df_v3['year'] == 2010]")
division_2010 = mega_df_v3[mega_df_v3['year'] == 2010]
print(f"   → {len(division_2010)} units in 2010")

# 2. Track units that changed over time
print("\n2. Track units that changed:")
print("   changed_units = mega_df_v3[mega_df_v3['if_changed'] == True]")
changed_units = mega_df_v3[mega_df_v3['if_changed'] == True]
print(f"   → {changed_units['id'].nunique()} unique units with changes")

# 3. Get history of a specific unit
print("\n3. Get history of a specific unit by name:")
print("   unit_history = luf.get_unit_history(mega_df_v3, unit_name='Warszawa')")
warsaw = luf.get_unit_history(mega_df_v3, unit_name='Warszawa')
print(f"   → Found {len(warsaw)} records for 'Warszawa'")

# 4. Get changes summary
print("\n4. Get summary of changes by year:")
print("   changes_summary = luf.get_changes_summary(mega_df_v3)")
changes_summary = luf.get_changes_summary(mega_df_v3)
print(f"   → {len(changes_summary)} years with changes")

# 5. Show sample of a changed unit with notes
print("\n5. Sample of change notes:")
sample = mega_df_v3[(mega_df_v3['if_changed'] == True) & (mega_df_v3['year'] == 2010)].iloc[0]
print(f"   Unit: {sample['NAZWA']} (ID: {sample['id']})")
print(f"   Changed in year: {int(sample['when_changed'])}")
print(f"   Notes: {sample['notes']}")

print("\n" + "=" * 60)
print("Mega DataFrame column summary:")
print("=" * 60)
print(mega_df_v3.dtypes)

TERYT HARMONIZATION - USAGE EXAMPLES

1. Get administrative division for a specific year:
   division_2010 = mega_df_v3[mega_df_v3['year'] == 2010]
   → 4105 units in 2010

2. Track units that changed:
   changed_units = mega_df_v3[mega_df_v3['if_changed'] == True]
   → 556 unique units with changes

3. Get history of a specific unit by name:
   unit_history = luf.get_unit_history(mega_df_v3, unit_name='Warszawa')
   → Found 100 records for 'Warszawa'

4. Get summary of changes by year:
   changes_summary = luf.get_changes_summary(mega_df_v3)
   → 25 years with changes

5. Sample of change notes:
   Unit: Olszyna (ID: 0210053)
   Changed in year: 2005
   Notes: {'number_of_changes': 1, 'changes': ['TYPE_CHANGE: rural -> urban-rural', 'DESIGNATION_CHANGE: gmina wiejska -> gmina miejsko-wiejska']}

Mega DataFrame column summary:
year              int64
WOJ              object
POW              object
GMI              object
RODZ             object
id               object
NAZWA            

In [ ]:
mega_df_v3.to_csv(geo_root / "teryt_df.csv", index=False, encoding="utf-8")

In [35]:
terc_changes

,TypKorekty,WojPrzed,PowPrzed,GmiPrzed,RodzPrzed,NazwaPrzed,NazwaDodatkowaPrzed,StanPrzed,WojPo,PowPo,...,WyodrebnionoZIdentyfikatora2,WyodrebnionoZIdentyfikatora3,WlaczonoDoIdentyfikatora1,WlaczonoDoIdentyfikatora2,WlaczonoDoIdentyfikatora3,StanPo,id_before,id_after,year_before,year_after
0,M,02,20,02,2,Prusice,gmina wiejska,1999-01-01,02,20,...,NaN,NaN,NaN,NaN,NaN,2000-01-01,0220022,0220023,1999,2000
1,D,00,00,00,0,None,None,1999-01-01,02,20,...,NaN,NaN,NaN,NaN,NaN,2000-01-01,0000000,0220024,1999,2000
2,D,00,00,00,0,None,None,1999-01-01,02,20,...,NaN,NaN,NaN,NaN,NaN,2000-01-01,0000000,0220025,1999,2000
3,M,06,18,12,2,Tyszowce,gmina wiejska,1999-01-01,06,18,...,NaN,NaN,NaN,NaN,NaN,2000-01-01,0618122,0618123,1999,2000
4,D,00,00,00,0,None,None,1999-01-01,06,18,...,NaN,NaN,NaN,NaN,NaN,2000-01-01,0000000,0618124,1999,2000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
561,D,00,00,00,0,None,None,2023-01-01,30,08,...,NaN,NaN,NaN,NaN,NaN,2024-01-01,0000000,3008064,2023,2024
562,D,00,00,00,0,None,None,2023-01-01,30,08,...,NaN,NaN,NaN,NaN,NaN,2024-01-01,0000000,3008065,2023,2024
563,M,30,28,04,2,Mieścisko,gmina wiejska,2023-01-01,30,28,...,NaN,NaN,NaN,NaN,NaN,2024-01-01,3028042,3028043,2023,2024
564,D,00,00,00,0,None,None,2023-01-01,30,28,...,NaN,NaN,NaN,NaN,NaN,2024-01-01,0000000,3028044,2023,2024
